In [15]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, roc_auc_score, classification_report


# 🔹 1. DATASET
data = pd.DataFrame({
    "user_id": [101,102,103,104,105,106,107,108,109,110,111,112,113,114,115],
    "course_id": ["C1","C2","C3","C1","C4","C2","C3","C5","C4","C1","C2","C5","C3","C4","C1"],
    "time_spent": [12.5,8.0,15.2,5.5,20.0,7.5,18.0,10.0,6.5,14.0,9.0,16.5,11.0,4.5,13.0],
    "quiz_score": [78,65,88,50,92,60,85,70,55,80,68,90,72,45,82],
    "assignments_completed": [5,3,6,2,8,3,7,4,2,6,4,7,5,1,6],
    "interactions": [30,20,40,10,50,18,45,25,15,35,22,48,28,8,33],
    "category": ["Data Science","Programming","AI","Data Science","Machine Learning",
                 "Programming","AI","Cloud Computing","Machine Learning","Data Science",
                 "Programming","Cloud Computing","AI","Machine Learning","Data Science"],
    "completion": [1,0,1,0,1,0,1,1,0,1,0,1,1,0,1]
})


print("\nDATASET SAMPLE:\n", data.head())


# 🔹 2. FEATURES & TARGET
X = data.drop("completion", axis=1)
y = data["completion"]


# 🔹 3. PREPROCESSING
num_features = ["time_spent","quiz_score","assignments_completed","interactions"]
cat_features = ["course_id","category"]

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler", StandardScaler())
    ]), num_features),

    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_features)
])


# 🔹 4. STRATIFIED SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


# =========================================================
# 🔵 MODEL 1: LOGISTIC REGRESSION
# =========================================================

lr_model = Pipeline([
    ("prep", preprocessor),
    ("lr", LogisticRegression())
])

lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)
lr_prob = lr_model.predict_proba(X_test)[:,1]

lr_acc = accuracy_score(y_test, lr_pred)
lr_auc = roc_auc_score(y_test, lr_prob)

print("\n LOGISTIC REGRESSION RESULTS")
print("Accuracy:", round(lr_acc,2))
print("ROC-AUC:", round(lr_auc,2))
print(classification_report(y_test, lr_pred))


# =========================================================
# 🟢 MODEL 2: RANDOM FOREST
# =========================================================

rf_model = Pipeline([
    ("prep", preprocessor),
    ("rf", RandomForestClassifier(n_estimators=100, random_state=42))
])

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:,1]

rf_acc = accuracy_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_prob)

print("\n RANDOM FOREST RESULTS")
print("Accuracy:", round(rf_acc,2))
print("ROC-AUC:", round(rf_auc,2))
print(classification_report(y_test, rf_pred))


# =========================================================
# 🔚 FINAL COMPARISON
# =========================================================

print("\n FINAL COMPARISON")

print("Logistic Regression -> Accuracy:", round(lr_acc,2), " ROC-AUC:", round(lr_auc,2))
print("Random Forest       -> Accuracy:", round(rf_acc,2), " ROC-AUC:", round(rf_auc,2))

if rf_auc > lr_auc:
    print("\nBEST MODEL: Random Forest")
else:
    print("\n BEST MODEL: Logistic Regression")


DATASET SAMPLE:
    user_id course_id  time_spent  quiz_score  assignments_completed  \
0      101        C1        12.5          78                      5   
1      102        C2         8.0          65                      3   
2      103        C3        15.2          88                      6   
3      104        C1         5.5          50                      2   
4      105        C4        20.0          92                      8   

   interactions          category  completion  
0            30      Data Science           1  
1            20       Programming           0  
2            40                AI           1  
3            10      Data Science           0  
4            50  Machine Learning           1  

 LOGISTIC REGRESSION RESULTS
Accuracy: 1.0
ROC-AUC: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         2

    accuracy                           1.00   